# ARKHER — nó Kaggle (Linux + GPU)

Roda o **mesmo `agent.py`, `dsos_core.py` e `gerar3d.py` do repositório** em Linux com **P100/T4**.

1. Settings → Accelerator → **GPU**
2. Settings → Internet → **On**
3. Add-ons → Secrets → nome `TS_KEY`, valor = sua chave do Tailscale
   (login.tailscale.com/admin/settings/keys, **Reusable**)
4. **Run All**. A última célula imprime os endereços para colar no site (aba VM, aba DsOS e aba 3D).

A sessão do Kaggle dura ~9-12 h (o do GitHub Actions, ~6 h). O notebook precisa ficar rodando —
o estado (`/kaggle/working`) sobrevive enquanto a sessão vive, e é salvo a cada 2 min.

**3D:** com GPU, o TripoSR (imagem→3D, malha limpa) e o Shap-E (texto→3D) rodam de verdade aqui.
O padrão instala o Shap-E; o TripoSR fica a um `INSTALAR_TRIPOSR = True` (a instalação leva ~5 min).
## O que este nó faz hoje
- **agente** (8765): comandos, arquivos, tela, entrada, piloto
- **DsOS** (8766): desktop Linux ao vivo
- **3D**: `gerar3d.py` (Shap-E / TripoSR) → `.glb`
- **modelos de ponta do HF** (`hf_hub.py`, rota `/infer`): texto/código, imagem/textura,
  profundidade, remover fundo, upscale, áudio→texto — rodando na GPU do Kaggle, de graça
- **treino LoRA** (`/treinar`): pega o dataset exportado do site (aba Cérebro) e treina
  um modelo seu em cima de Qwen2.5-Coder. Depois o site usa `gpu:local:<nome>`

Se a sessão cair, o `religa_kaggle.py` (rodando na VM do Actions ou no Termux) empurra
este notebook de novo automaticamente. Lembre: a cota é ~30 h de GPU por semana.


In [ ]:
# 1) Tailscale (userspace: o Kaggle nao da /dev/net/tun)
import os, subprocess, time
from kaggle_secrets import UserSecretsClient
TS = UserSecretsClient().get_secret("TS_KEY")
subprocess.run("curl -fsSL https://tailscale.com/install.sh | sh", shell=True, capture_output=True)
subprocess.Popen(["tailscaled", "--tun=userspace-networking", "--socks5-server=localhost:1055",
                  "--state=/kaggle/working/ts.state"], stdout=open("/kaggle/working/tailscaled.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
r = subprocess.run(["tailscale", "up", "--authkey=" + TS, "--hostname=arkher-kaggle",
                    "--accept-dns=false"], capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-500:])
ip = subprocess.run(["tailscale", "ip", "-4"], capture_output=True, text=True).stdout.strip().split("\n")[0]
assert ip, "sem IP do Tailscale — a chave expirou? marque Reusable no painel"
print("Tailscale OK:", ip)

In [ ]:
# 2) baixa os arquivos do repo (sempre a versao atual) e sobe o agente na 8765
import subprocess, time, urllib.request, os
REPO_RAW = "https://raw.githubusercontent.com/WhiteXz7/ARKHER-AI/main"
for f in ("agent.py", "dsos_core.py", "gerar3d.py", "ws_min.py", "hf_hub.py"):
    urllib.request.urlretrieve(REPO_RAW + "/" + f, "/kaggle/working/" + f)
    print("baixado:", f)
os.makedirs("/kaggle/working/arkher_state/work", exist_ok=True)
env = dict(os.environ, ARKHER_STATE="/kaggle/working/arkher_state", ARKHER_PORT="8765")
subprocess.Popen(["python3", "/kaggle/working/agent.py"], env=env,
                 stdout=open("/kaggle/working/agent.log", "w"), stderr=subprocess.STDOUT)
for i in range(20):
    time.sleep(2)
    try:
        import json; h = json.load(urllib.request.urlopen("http://127.0.0.1:8765/health", timeout=5))
        print("agente no ar:", h.get("os"), "|", h.get("cpu"), "|", h.get("ram_gb"), "GB"); break
    except Exception as e:
        print("esperando...", e)
else:
    raise SystemExit("agente nao subiu: " + open("/kaggle/working/agent.log").read()[-800:])

In [ ]:
# 2b) PILHA DE MODELOS DE PONTA (Hugging Face) — texto, imagem, audio, treino
# O Kaggle ja vem com torch + CUDA. Aqui entra o resto do hf_hub.py.
# Isto e o que faz o no valer mais que a API gratis do HF (que hoje cobra por token):
# o modelo roda na SUA GPU, quantizado em 4 bits pra caber nos 16 GB do T4.
import os, subprocess, time
INSTALAR_HF   = True     # <- False pula (se voce so quer 3D)
INSTALAR_TREINO = True   # trl/peft/datasets: o que treina o SEU LoRA
os.environ["HF_HOME"] = "/kaggle/working/hf"     # pesos baixados ficam salvos
os.makedirs("/kaggle/working/hf", exist_ok=True)

# token do HF e opcional: serve pra modelo com licenca (gate) e download mais rapido.
# Kaggle > Add-ons > Secrets > nome HF_TOKEN (se voce tiver). Sem ele funciona igual.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN lido dos secrets")
except Exception as e:
    print("sem HF_TOKEN nos secrets (ok):", type(e).__name__)

if INSTALAR_HF:
    pk = "transformers diffusers accelerate bitsandbytes sentencepiece einops safetensors pillow"
    if INSTALAR_TREINO:
        pk += " trl peft datasets"
    print("instalando:", pk)
    t0 = time.time()
    r = subprocess.run("pip install -q " + pk, shell=True, capture_output=True, text=True)
    print("rc =", r.returncode, "| %.1f min" % ((time.time()-t0)/60))
    if r.returncode: print((r.stderr or r.stdout)[-1500:])
    # unsloth: opcional, treina ~2x mais rapido no T4. Se falhar, o hf_hub cai no trl.
    try:
        r2 = subprocess.run("pip install -q unsloth", shell=True, capture_output=True, text=True, timeout=900)
        print("unsloth rc =", r2.returncode)
    except Exception as e:
        print("unsloth nao instalou (sem problema):", type(e).__name__)


In [ ]:
# 2c) O QUE ESTE NO PODE RODAR (catalogo real, lido do proprio hf_hub.py)
import json, subprocess, os, time
os.environ["ARKHER_STATE"] = "/kaggle/working/arkher_state"
time.sleep(3)
p = subprocess.run(["python3", "/kaggle/working/hf_hub.py", "--lista"],
                   text=True, capture_output=True, env=os.environ)
try:
    cat = json.loads(p.stdout)
except Exception:
    print("nao consegui ler o catalogo. saida:", (p.stdout or p.stderr)[-1500:]); cat = {}
amb = cat.get("ambiente", {}) or {}
g = amb.get("gpu", {}) or {}
print("GPU:", g.get("gpu") or "(nenhuma)", "| VRAM: %.1f GB" % (g.get("vram_gb") or 0),
      "| RAM:", amb.get("ram_gb"), "GB")
print("prontas:", [k for k, v in (cat.get("tarefas") or {}).items() if v.get("pronto")])
print("faltando:", {k: v.get("falta") for k, v in (cat.get("tarefas") or {}).items() if not v.get("pronto")})
print()
print("modelos de ponta disponiveis neste no (os que cabem na VRAM):")
for m in cat.get("modelos", []):
    print("  %-11s %-42s ~%s GB  %s" % (m["tarefa"], m["id"], m["vram_gb"], m["papel"]))
print()
print("treino LoRA: bases", [b["id"] for b in cat.get("bases_treino", [])])
print("rotas:", json.dumps(cat.get("rotas", {}), ensure_ascii=False, indent=1))


In [ ]:
# 2d) CEREBRO DO NO + TREINO AUTOMATICO + VIGIA
#  - o site manda o que aprendeu (licoes e preferencias) -> /dataset
#  - quando juntar amostra nova suficiente, ele TREINA SOZINHO (LoRA ou DPO) e o
#    modelo vira "local:<nome>" no catalogo do site
#  - o vigia religa o agent.py se cair e pinga o proprio no (Kaggle mata por 20 min parado)
import os, subprocess, time, urllib.request, json
BASE = "http://127.0.0.1:8765"
os.environ["ARKHER_STATE"] = "/kaggle/working/arkher_state"

def pedir(caminho, dados=None, timeout=20):
    req = urllib.request.Request(BASE + caminho,
                                 data=(json.dumps(dados).encode() if dados else None),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode())

# 1) liga o treino automatico (usa o que chegar em /dataset) e mostra o que ja tem
try:
    print("treino automatico:", pedir("/dataset/auto", {"ligado": True, "min_novos": 40,
                                                        "min_total": 40, "passos": 60}))
except Exception as e:
    print("nao ligou o treino automatico:", e)
try:
    print("dataset do no:", pedir("/dataset"))
except Exception as e:
    print("sem dataset ainda:", e)

# 2) vigia: mantem o no acordado e de pe (rode esta celula e deixe a sessao viva)
def vigia(segundos=1800):
    fim = time.time() + segundos
    while time.time() < fim:
        time.sleep(180)
        try:
            pedir("/health", timeout=10)          # ping: evita o "idle" de 20 min do Kaggle
        except Exception:
            print("agente caiu? subindo de novo...", flush=True)
            subprocess.Popen(["python3", "/kaggle/working/agent.py"],
                             env=dict(os.environ, ARKHER_STATE="/kaggle/working/arkher_state",
                                      ARKHER_PORT="8765"),
                             stdout=open("/kaggle/working/agent.log", "a"),
                             stderr=subprocess.STDOUT)
            time.sleep(10)
        try:
            d = pedir("/dataset", timeout=10)
            a = d.get("auto", {})
            print(time.strftime("%H:%M"), "amostras:", d.get("total"), "| treinos:", a.get("jobs"),
                  "| treinados:", (d.get("adapters") or [])[-2:], flush=True)
        except Exception as e:
            print("aviso:", e, flush=True)

vigia(segundos=60 * 60 * 11)   # ~11 h (a sessao do Kaggle dura ~12 h)


In [ ]:
# 3) ambiente grafico do DsOS (Xvfb + gerenciador de janelas + captura + entrada). 1-3 min.
import subprocess
print("instalando ambiente grafico...")
subprocess.run("apt-get update -qq", shell=True, capture_output=True)
pk = "xvfb x11-utils xdotool imagemagick ffmpeg openbox tint2 xterm x11-xserver-utils fonts-dejavu pcmanfm mousepad"
r = subprocess.run("DEBIAN_FRONTEND=noninteractive apt-get install -y -qq " + pk, shell=True,
                   capture_output=True, text=True)
print("apt rc =", r.returncode)
import shutil
for b in ["Xvfb", "xdotool", "ffmpeg", "openbox", "xterm", "import"]:
    print(" ", b, "->", shutil.which(b) or "FALTOU")

In [ ]:
# 4) sobe o DsOS Core na 8766 (mesmo arquivo do repo) — desktop + tela + entrada
import os, subprocess, time, urllib.request, json, shutil
os.makedirs("/kaggle/working/dsos", exist_ok=True)
env = dict(os.environ, DSOS_PORT="8766", DSOS_ROOT="/kaggle/working/dsos", DSOS_BACKEND="kaggle",
           DSOS_W="1280", DSOS_H="720", DSOS_AUTOBOOT="1", DSOS_DISPLAY=":77", DISPLAY=":77")
subprocess.Popen(["python3", "/kaggle/working/dsos_core.py"], env=env,
                 stdout=open("/kaggle/working/dsos.log", "w"), stderr=subprocess.STDOUT)
time.sleep(12)
print(open("/kaggle/working/dsos.log").read()[-1200:])
try:
    c = json.load(urllib.request.urlopen("http://127.0.0.1:8766/capacidades", timeout=15))
    print("DsOS: tela=", c.get("tela"), "captura=", c.get("captura"), "entrada=", c.get("entrada"),
          "gpu_compute=", c.get("gpu_compute"))
    print("ws_min.py (WebSocket da tela):", "ok" if os.path.exists("/kaggle/working/ws_min.py") else "faltando -> cai pro polling")
except Exception as e:
    print("DsOS ainda nao respondeu:", e, "— veja /kaggle/working/dsos.log")

In [ ]:
# 5) motores 3D de verdade (o mesmo gerar3d.py do repo). Com GPU: Shap-E (texto->3D) agora,
#    TripoSR (imagem->3D, malha limpa) se INSTALAR_TRIPOSR = True. ~5 min a mais.
import os, subprocess, time
INSTALAR_TRIPOSR = False        # <- mude pra True se quiser a melhor malha (compila o torchmcubes)
os.environ["HF_HOME"] = "/kaggle/working/hf"        # pesos ficam aqui (nao rebaixa a cada run)
os.environ["ARKHER_STATE"] = "/kaggle/working/arkher_state"
def rodar(cmd):
    p = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(p.stdout[-2500:] or p.stderr[-1200:]); return p.returncode
print("=== dependencias base ===")
rodar("pip -q install trimesh pillow pyglet fast_simplification 2>&1 | tail -2")
print("=== instalando o Shap-E (pip puro, sem compilar) ===")
rc = rodar("python3 /kaggle/working/gerar3d.py --instalar shape")
if INSTALAR_TRIPOSR:
    print("=== instalando o TripoSR (clona o repo + torchmcubes) ===")
    RC = rodar("python3 /kaggle/working/gerar3d.py --instalar triposr")
print("=== o que este no tem agora ===")
rodar("python3 /kaggle/working/gerar3d.py --lista")

In [ ]:
# 6) teste de verdade: gera um modelo e mostra o render aqui mesmo
import os, subprocess, json
from IPython.display import Image, display
os.environ["ARKHER_STATE"] = "/kaggle/working/arkher_state"
os.environ["HF_HOME"] = "/kaggle/working/hf"
p = subprocess.run(["python3", "/kaggle/working/gerar3d.py", "--prompt", "um cogumelo vermelho", "--engine", "shape"],
                   text=True, capture_output=True, timeout=3600)
print(p.stdout[-3000:] or p.stderr[-1500:])
meta = {}
for l in p.stdout.splitlines():
    if l.startswith("ARKHER3D_OK "): meta = json.loads(l[12:])
if meta.get("preview_abs") and os.path.exists(meta["preview_abs"]):
    display(Image(meta["preview_abs"], width=420))
    print("glb:", meta["glb_abs"], "| faces:", meta.get("faces"), "|", meta.get("segundos"), "s")
    print("baixe pela aba 3D do site, ou aqui: /kaggle/working/arkher_state/work/3d/")
else:
    print("nao gerou — o log acima diz o motivo (dependencia faltando? VRAM?) — sem GPU use --engine procedural")

In [ ]:
# 7) enderecos para colar no site — e mantem o no vivo
print('  aba Cerebro > treinar/estudar: http://%s:8765/infer (catalogo de modelos de ponta)' % ip)
print('=' * 58)
print('  aba VM    > URL do no Kaggle:  http://%s:8765' % ip)
print('  aba DsOS  > Conectar:          http://%s:8766' % ip)
print('  aba 3D    > No de calculo:     (escolha "Kaggle (GPU)")')
print('  aba Cerebro > treinar/estudar: http://%s:8765/infer (catalogo de modelos de ponta)' % ip)
print('=' * 58)
print('Catalogo de modelos 3D prontos na GPU:'
      ' http://%s:8766/capacidades' % ip)
print('Deixe esta celula rodando. Stop encerra o no (o estado ja foi salvo).')
import time, urllib.request
while True:
    time.sleep(300)
    try: urllib.request.urlopen('http://127.0.0.1:8765/snapshot', data=b'{}', timeout=10)
    except Exception: pass